In [ ]:
import sys
from pathlib import Path

import pandas as pd

try:
    import dedekind
    from dedekind import (
        Activity,
        Dual,
        Ensemble,
        LinearChoice,
        critical_path_schedule,
        dual_derivative,
        solve_mixed_integer_plan,
    )
except ModuleNotFoundError:
    candidate_roots = [Path.cwd(), *Path.cwd().parents]
    for root in candidate_roots:
        candidate = root / "python"
        if (candidate / "dedekind").exists():
            sys.path.insert(0, str(candidate))
            break
    import dedekind
    from dedekind import (
        Activity,
        Dual,
        Ensemble,
        LinearChoice,
        critical_path_schedule,
        dual_derivative,
        solve_mixed_integer_plan,
    )

print("Imported formal-tier primitive: Ensemble")
print("Operators: | (union)  & (intersection)  - (difference)  <= (subset)")
print("Methods:   .comprehension()  .restrict()  .map_to()  .realize()  .to_dataframe()")
print("Extra middleware: Dual numbers, mixed-integer planning, critical-path scheduling")

## Part 1: Extensional Ensembles and Operator Syntax

Finite sets defined by explicit enumeration. Python operator syntax (`|`, `&`, `-`) provides
the easy surface; named methods (`.union()`, `.intersection()`, `.difference()`) are the
formal aliases (issue #241 operator/method parity).

In [ ]:
A = Ensemble.from_members(1, 2, 3, 4, 5)
B = Ensemble.from_members(3, 4, 5, 6, 7)

U = A | B   # union:        A ∪ B
I = A & B   # intersection: A ∩ B
D = A - B   # difference:   A \ B

print(f"A        = {A}")
print(f"B        = {B}")
print(f"A ∪ B    = {U.realize()}")
print(f"A ∩ B    = {I.realize()}")
print(f"A \\ B   = {D.realize()}")
print(f"A ⊆ A∪B? {A <= U}")
print(f"B ⊆ A∩B? {B <= I}")

## Part 2: Intensional Definitions via Comprehension

Set-builder notation: `Ensemble.comprehension(universe, predicate)` mirrors the mathematical
$\{ x \in U \mid P(x) \}$ form. The definition stays intentional until `.realize()` is called —
the explicit realization boundary.

In [ ]:
# {n ∈ [1..30] | n ≡ 0 (mod 2)}  —  evens
E = Ensemble.comprehension(range(1, 31), lambda n: n % 2 == 0)

# {n ∈ [1..30] | n ≡ 1 (mod 2)}  —  odds
O = Ensemble.comprehension(range(1, 31), lambda n: n % 2 == 1)

# Comprehensions partition the universe: E ∪ O = {1..30}, E ∩ O = ∅
EuO = (E | O).realize()
EiO = (E & O).realize()

print(f"Evens  E = {E.realize()}")
print(f"Odds   O = {O.realize()}")
print(f"E ∪ O    = {EuO}  (len={len(EuO)})")
print(f"E ∩ O    = {EiO}  (∅ ↔ empty list: {EiO == []})")

## Part 3: Morphism Image

`.map_to(f)` applies a morphism $f : S \to T$ to produce the image set $f(S) = \{ f(x) \mid x \in S \}$.
Chaining `.restrict().map_to()` corresponds to a restricted morphism.

In [ ]:
# Image under squaring morphism: Q = {n² | n ∈ E}
Q = E.map_to(lambda n: n * n)

# Restrict then map: squares of evens ≤ 10
Q_small = E.restrict(lambda n: n <= 10).map_to(lambda n: n * n)

print(f"Squares of evens Q        = {Q.realize()}")
print(f"Squares of evens ≤ 10     = {Q_small.realize()}")

# DataFrame rendering
print()
print("Evens as DataFrame:")
print(E.to_dataframe("n"))

In [ ]:
# --- Assertions ---
assert (A | B).realize() == [1, 2, 3, 4, 5, 6, 7]
assert (A & B).realize() == [3, 4, 5]
assert (A - B).realize() == [1, 2]

evens = E.realize()
odds  = O.realize()
assert len(evens) == 15
assert len(odds)  == 15
assert sorted(evens + odds) == list(range(1, 31))

squares = Q.realize()
assert len(squares) == 15
assert squares[0] == 4    # 2² = 4
assert squares[-1] == 900 # 30² = 900

assert A <= (A | B)
assert not (B <= (A & B))

print("formal-tier invariants verified")
print("notebook-04-ok")

In [ ]:
# Part 4: Minimal DSL v1 (Agent-Oriented)
#
# This section now behaves more like an executable report than an IDE scratchpad.
# The plumbing lives in the middleware shim; the notebook calls it to surface
# actionable optimization and scheduling insights.
#
# Math-forward ingredients:
# - dual numbers for first-order derivative signals (`#177`, `#272`)
# - constrained planning / Lagrange-adjacent optimization hooks (`#231`)
# - DAG scheduling / critical-path reasoning (`#339`, with graph-theory bridge to `#329`)
#
# Report questions:
# 1. What does a local dual-number derivative say about the relaxed objective?
# 2. Which mixed-integer implementation batch maximizes value under a budget?
# 3. Which dependency chain is the critical path for delivery?

In [ ]:
def relaxed_utility(x):
    # FIXME #231: replace penalty-style objective tuning with a proper Lagrange toolkit.
    return 11 * x - 1.5 * (x ** 2) - 0.75 * ((2 * x - 5) ** 2)

value_at_two, slope_at_two = dual_derivative(relaxed_utility, 2.0)

implementation_batch = solve_mixed_integer_plan(
    [
        LinearChoice("pivot-kernel", value=12.0, cost=5.0, max_units=1),
        LinearChoice("join-audit", value=8.0, cost=3.0, max_units=1),
        LinearChoice("notebook-report", value=5.0, cost=2.0, max_units=2),
        LinearChoice("flow-bridge", value=7.0, cost=4.0, max_units=1),
    ],
    budget=9.0,
)

delivery_schedule = critical_path_schedule(
    [
        Activity("spec", duration=2.0),
        Activity("dual-kernel", duration=2.0, depends_on=("spec",)),
        Activity("mip-pass", duration=3.0, depends_on=("dual-kernel",)),
        Activity("report", duration=1.0, depends_on=("mip-pass",)),
        Activity("review", duration=2.0, depends_on=("report",)),
        Activity("graph-bridge", duration=2.0, depends_on=("spec",)),
    ]
)

selected = implementation_batch["decision_table"]
selected = selected[selected["quantity"] > 0].reset_index(drop=True)

derivative_report = pd.DataFrame(
    [
        {
            "probe": "relaxed_utility(x)",
            "x": 2.0,
            "value": value_at_two,
            "d_dx": slope_at_two,
            "insight": "positive slope => more budget pressure can still improve objective locally",
        }
    ]
)

summary_report = pd.DataFrame(
    [
        {
            "metric": "best_value",
            "value": implementation_batch["objective"],
        },
        {
            "metric": "budget_used",
            "value": implementation_batch["cost"],
        },
        {
            "metric": "project_duration",
            "value": delivery_schedule["project_duration"],
        },
    ]
)

print("dual-number local sensitivity")
display(derivative_report)

print("selected mixed-integer plan")
display(selected)

print("critical path schedule")
display(delivery_schedule["schedule"])

print("summary")
display(summary_report)

assert slope_at_two > 0
assert implementation_batch["cost"] <= implementation_batch["budget"]
assert delivery_schedule["critical_path"] == ["spec", "dual-kernel", "mip-pass", "report", "review"]

print("formal optimization report verified")